# Caso de referencia RESUELTO — Equipo editorial multiagente
**Semana 1 · Sesión 14 · LangGraph + Groq · Lectura guiada y ejecución en Colab**

Este notebook reimplementa con **LangGraph** el ejemplo de reflexión y revisión especializada de `L3_Reflection_and_Blogpost_Writing.ipynb` (originalmente en AutoGen).

> ⚠️ Es una **demostración guiada**, NO la entrega evaluada. Su objetivo es que entiendas *cómo* se coordina un equipo de agentes antes de diseñar el tuyo.

### 🎯 Al terminar podrás
1. Explicar qué hace cada agente del equipo editorial y **qué parte del estado lee y escribe**.
2. Leer un `StateGraph`: estado, nodos, aristas fijas y **arista condicional**.
3. Ejecutar el grafo con Groq (o sin costo, en modo simulado) y **seguir la ruta** que tomó.
4. Medir el **costo de coordinación** (llamadas LLM y latencia) y distinguir lo que el ejemplo demuestra de lo que **no** demuestra.

### 🗺️ Cómo está organizado
| # | Sección | Qué haces |
|---|---|---|
| 1 | Problema y arquitectura | Lees el caso y el diagrama |
| 2 | Preparar Colab y Groq | Configuras la clave y el modo |
| 3 | Estado compartido | Defines el “tablero” común de los agentes |
| 4 | Interfaz con el LLM | Un único punto de contacto con Groq |
| 5 | Nodos (agentes) | Un rol = una función |
| 6 | Grafo | Conectas los nodos y lo visualizas |
| 7 | Ejecución y evidencia | Corres el flujo y mides costo |
| 8 | Análisis arquitectónico | Discutes alcances y límites |
| 9 | Transferencia | El reto que sí se entrega |
| 10 | Fuentes | Referencias |

### 🔣 Convenciones de este notebook
- 🧠 **Concepto**: idea que hay que entender antes del código.
- ▶️ **Ejecutar**: celda que debes correr.
- 🤔 **Pausa**: pregunta para discutir en clase antes de seguir.
- ⚠️ **Cuidado**: error frecuente o limitación importante.

## 1. Problema y requisitos de aceptación

Una organización desea publicar una entrada breve de blog sobre *cómo pequeños negocios pueden usar IA de forma responsable*.

| Requisito | ¿Qué agente lo cuida? |
|---|---|
| Título + cuerpo de **máximo 130 palabras**, tono claro | Escritor (redacta) y Evaluador (verifica) |
| **Sin cifras ni beneficios inventados**, sin fuentes no verificadas | Revisor legal |
| **Advertencia breve sobre privacidad** | Revisor ético y Revisor legal |
| Revisión **SEO**, **legal** y **ética** antes de publicar | Las aristas del grafo lo garantizan |
| Consolidar los hallazgos | Metarrevisor |
| Decidir si se acepta o se reescribe | Evaluador |
| **Máximo 3 versiones** del borrador (para no crear bucles infinitos) | Función de ruta `ruta_evaluacion` |

> 🧠 Observa que algunos requisitos los cumple **un agente** (con su prompt) y otros los cumple **la arquitectura** (con aristas y límites). Esa distinción es el corazón del diseño multiagente.

### Arquitectura del flujo

```
                 ┌───────────── Revisión especializada (secuencial) ─────────────┐
 START ──► Escritor ──► Revisor SEO ──► Revisor legal ──► Revisor ético ──► Metarrevisor
              ▲                                                                  │
              │                                                                  ▼
              │  "reescribir"  (veredicto REVISAR y versión < 3)            Evaluador
              └──────────────────────────────── ruta_evaluacion ◄──────────────┘
                                                      │
                                                      └──► "fin" ──► END
```

**Cómo se lee:**
1. El **Escritor** produce la versión 1 del borrador.
2. Los tres **revisores** comentan el borrador, uno después del otro (orden fijo).
3. El **Metarrevisor** consolida los tres comentarios en una lista priorizada.
4. El **Evaluador** juzga el borrador actual frente a los requisitos y al feedback consolidado, y responde `APROBAR` o `REVISAR`.
5. La función `ruta_evaluacion` decide: si dice `REVISAR` y aún hay margen de versiones, vuelve al Escritor (que ahora sí incorpora el feedback); si no, termina.

| Dato | Valor |
|---|---|
| Agentes (nodos LLM) | 6 |
| Llamadas LLM por iteración | 6 |
| Versiones máximas del borrador | 3 → peor caso: **18 llamadas** |

> ⚠️ **`END` no significa “aprobado”.** Solo significa que el flujo terminó. El veredicto real queda guardado en el estado y hay que leerlo.

> 🧠 Los revisores son **secuenciales** en esta versión por motivos de cuota y trazabilidad. En la sección 8 discutimos cómo paralelizarlos.

🤔 **Pausa (2 min):** antes de ver código, responde:
- ¿Qué pasaría si el Evaluador evaluara *después* de que el Escritor incorpore el feedback? ¿Cambiaría el número de llamadas?
- ¿Cuál de estos requisitos **no** puede garantizar ningún prompt, por bueno que sea?

## 2. Preparar Google Colab y Groq

### Paso a paso para la clave
1. Crea una clave en la consola de Groq.
2. En Colab, abre el panel 🔑 **Secrets** (barra lateral izquierda).
3. Crea un secreto llamado exactamente `GROQ_API_KEY` y pega la clave como valor.
4. Activa el interruptor **“Acceso del notebook”** para este notebook.

> ⚠️ **Nunca** pegues la clave en una celda de código ni la incluyas en la entrega.

### Dos modos de ejecución
| Modo | `MODO_SIMULADO` | Qué ocurre | Cuándo usarlo |
|---|---|---|---|
| Simulado | `True` | Respuestas fijas, sin llamar a Groq, sin costo | Para entender el flujo o practicar sin cuota |
| Real | `False` | Cada agente hace una llamada real al LLM | Para observar comportamiento y medir latencia |

> ⚠️ Las respuestas simuladas solo demuestran el **flujo**; **no son evaluaciones reales** de un LLM. El ejemplo tampoco realiza análisis jurídico real: solo identifica riesgos de redacción.

### ▶️ 2.1 Instalar dependencias
Se fijan rangos de versión para que el notebook se comporte igual en todos los equipos. `pandas>=2.2.3,<3` evita conflictos y permite instalar una rueda compatible con el intérprete.

In [1]:
# langgraph : construye y ejecuta el grafo de agentes (el "plano de control").
# groq      : cliente oficial para llamar a los modelos alojados en Groq (la "inferencia").
# pydantic  : contratos de datos (se usa a fondo en la sesión 15).
# pandas    : tablas para revisar resultados y costos.
%pip install -q "langgraph>=1,<2" "groq>=0.30" "pydantic>=2,<3" "pandas>=2.2.3,<3" "python-dotenv>=1,<2"

Note: you may need to restart the kernel to use updated packages.


### ▶️ 2.2 Configuración y conexión
Esta celda decide el modo de ejecución, elige el modelo y crea el cliente de Groq **solo si** vamos a hacer llamadas reales.

In [2]:
import os
import time

from dotenv import load_dotenv

load_dotenv()
print("Clave disponible:", bool(os.getenv("GROQ_API_KEY")))
from typing import TypedDict

from langgraph.graph import StateGraph, START, END   # piezas para construir el grafo
from IPython.display import Image, display, Markdown  # para mostrar el diagrama y el artículo

# ── 1. Modo de ejecución ──────────────────────────────────────────────────────
# True  → simulado: no llama a Groq ni consume tokens (ideal para practicar).
# False → real: cada agente llama al modelo de Groq.
MODO_SIMULADO = False

# ── 2. Modelo ─────────────────────────────────────────────────────────────────
# Se puede cambiar sin tocar el código definiendo la variable de entorno GROQ_MODEL.
MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")

# ── 3. Cliente de Groq (solo en modo real) ───────────────────────────────────
client = None
if not MODO_SIMULADO:
    from groq import Groq

    # En VS Code se lee la variable de entorno; en Colab se intenta leer también su Secret.
    key = os.getenv("GROQ_API_KEY")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("GROQ_API_KEY")
        except ImportError:
            pass
    if not key:
        raise ValueError("Configura GROQ_API_KEY en el entorno o en los Secrets de Colab.")
    client = Groq(api_key=key)

print("Modo:", "simulado (sin costo)" if MODO_SIMULADO else "Groq real", "| Modelo:", MODEL)

Clave disponible: True
Modo: Groq real | Modelo: openai/gpt-oss-120b


## 3. 🧠 El estado compartido: el “tablero” del equipo

En LangGraph los agentes **no se llaman entre sí**. Cada nodo:
1. **lee** el estado completo,
2. hace su trabajo,
3. **devuelve solo los campos que cambia** (un diccionario parcial).

LangGraph fusiona esa actualización con el estado y pasa el resultado al siguiente nodo. Por eso el estado funciona como un **contrato**: define qué información existe y quién es responsable de cada pieza.

| Campo | Tipo | Lo escribe | Para qué sirve |
|---|---|---|---|
| `tema`, `requisitos` | `str` | (entrada) | Lo que se pide; nadie lo modifica |
| `borrador` | `str` | Escritor | Texto actual del artículo |
| `version` | `int` | Escritor | Cuántas versiones van; controla el límite del bucle |
| `seo`, `legal`, `etica` | `str` | Cada revisor | Comentario de su especialidad |
| `metarevision` | `str` | Metarrevisor | Feedback consolidado para el Escritor |
| `veredicto` | `str` | Evaluador | `APROBAR` o `REVISAR` + justificación |
| `historial` | `list[str]` | Todos | Ruta real que siguió la ejecución (trazabilidad) |

> 🧠 **Propiedad de campos:** cada campo tiene un único dueño (salvo `historial`). Esto evita que un agente sobrescriba el trabajo de otro y es requisito para paralelizar más adelante.

In [ ]:
class EstadoBlog(TypedDict):
    # ── Entrada (no cambia durante la ejecución) ──
    tema: str            # tema del artículo
    requisitos: str      # criterios de aceptación que todos los agentes deben respetar
    # ── Producto principal ──
    borrador: str        # texto actual (título + cuerpo); lo reescribe el Escritor
    version: int         # contador de versiones; lo usa la ruta para limitar el bucle
    # ── Feedback especializado (un campo por revisor) ──
    seo: str
    legal: str
    etica: str
    metarevision: str    # síntesis priorizada de seo + legal + etica
    # ── Decisión y trazabilidad ──
    veredicto: str       # respuesta del Evaluador: empieza por APROBAR o REVISAR
    historial: list[str] # nombres de los nodos en el orden en que se ejecutaron


# Datos del problema: se cargan en el estado inicial (sección 7).
TEMA = "Cómo pueden usar la IA de forma responsable los pequeños negocios"
REQUISITOS = ("Redactar en español título y cuerpo de máximo 130 palabras; "
              "tono accesible, no inventar cifras, no prometer resultados garantizados; "
              "mencionar privacidad; no citar fuentes no verificadas.")
print("Estado definido con", len(EstadoBlog.__annotations__), "campos")

## 4. 🧠 Interfaz con el LLM: un único punto de contacto

Todos los agentes llaman al modelo a través de **una sola función**, `pedir_llm()`. Ventajas:
- Cambiar de modelo o de proveedor se hace en un solo lugar.
- Podemos **contar llamadas y medir latencia** de cada agente (evidencia para el análisis de costo).
- El modo simulado se implementa aquí, sin tocar los nodos.

Cada agente se distingue solo por su **rol** (mensaje `system`) y su **tarea** (mensaje `user`).

| Parámetro | Valor | Por qué |
|---|---|---|
| `temperature` | `0.2` | Poca variación creativa; respuestas más estables entre ejecuciones |
| `max_completion_tokens` | 120–350 según el agente | Controla costo y evita saturar el límite de tokens por minuto (TPM) |
| `reasoning_effort` | `"low"` | El modelo razona poco antes de responder: más rápido y barato |

> ⚠️ En modelos con razonamiento, si el agente gasta todo `max_completion_tokens` pensando, el texto de respuesta puede llegar **vacío**. La función lo detecta y avisa.

In [ ]:
# ── Registro de llamadas: evidencia para medir el costo de coordinación ──────
LLAMADAS: list[dict] = []   # una fila por llamada: agente, segundos, tokens

# ── Respuestas simuladas por agente (solo se usan si MODO_SIMULADO = True) ───
RESPUESTAS_SIMULADAS = {
    "escritor": ("IA responsable en pequeños negocios\n\n"
                 "La inteligencia artificial puede ayudar a organizar preguntas frecuentes "
                 "y generar borradores de mensajes. Antes de usarla, protege los datos de "
                 "clientes, verifica las respuestas y define quién supervisa cada decisión. "
                 "Empieza con tareas pequeñas y mide resultados sin prometer milagros."),
    "seo": "Incluir palabra clave en título. Usar un subtítulo claro. Evitar relleno.",
    "legal": "No afirmar cumplimiento normativo. Revisar tratamiento de datos personales.",
    "etica": "Incluir supervisión humana y advertir sobre sesgos y privacidad.",
    "metarrevisor": "Conservar lenguaje claro; fortalecer privacidad y supervisión humana.",
}
_evaluaciones_simuladas = 0   # para que el simulador muestre el bucle de reescritura

def respuesta_simulada(agente: str) -> str:
    # El Evaluador simulado pide REVISAR la primera vez y APROBAR después,
    # así se puede observar el ciclo de reflexión sin gastar tokens.
    global _evaluaciones_simuladas
    if agente == "evaluador":
        _evaluaciones_simuladas += 1
        if _evaluaciones_simuladas == 1:
            return "REVISAR: falta reforzar la advertencia de privacidad."
        return "APROBAR: requisitos fundamentales presentes."
    return RESPUESTAS_SIMULADAS.get(agente, "Revisión completada.")


def pedir_llm(agente: str, rol: str, tarea: str, max_tokens: int = 220) -> str:
    # Envía (rol, tarea) al modelo y devuelve el texto de la respuesta.
    #   agente     : nombre corto del nodo que llama (para el registro y la simulación)
    #   rol        : mensaje "system" → define la especialidad y las restricciones
    #   tarea      : mensaje "user"   → el material concreto a procesar
    #   max_tokens : tope de tokens generados por esta llamada
    inicio = time.perf_counter()

    if MODO_SIMULADO:
        texto, tokens = respuesta_simulada(agente), 0
    else:
        respuesta = client.chat.completions.create(
            model=MODEL,
            temperature=0.2,
            max_completion_tokens=max_tokens,
            reasoning_effort="low",
            messages=[{"role": "system", "content": rol},    # quién es el agente
                      {"role": "user",   "content": tarea}],  # qué debe hacer ahora
        )
        texto = (respuesta.choices[0].message.content or "").strip()
        uso = getattr(respuesta, "usage", None)
        tokens = getattr(uso, "completion_tokens", None) if uso else None

    segundos = round(time.perf_counter() - inicio, 2)
    LLAMADAS.append({"agente": agente, "segundos": segundos, "tokens_generados": tokens})

    if not texto:
        print(f"⚠️ {agente}: respuesta vacía (¿se agotó max_completion_tokens={max_tokens}?)")
    return texto


def registrar(estado: EstadoBlog, etiqueta: str) -> list[str]:
    # Devuelve una NUEVA lista con el nodo agregado al final.
    # No se usa .append() sobre la lista del estado: los nodos devuelven valores
    # nuevos en lugar de modificar el estado "por debajo".
    return estado["historial"] + [etiqueta]

print("Interfaz LLM lista · modo simulado:", MODO_SIMULADO)

## 5. 🧠 Implementación: un nodo por especialidad

Todos los nodos siguen **el mismo patrón de 3 pasos**:

```python
def nodo(estado):
    tarea = ...construir a partir de los campos que necesita leer...
    texto = pedir_llm(agente, ROL_DEL_AGENTE, tarea, max_tokens)
    return {"campo_propio": texto, "historial": registrar(estado, "nodo")}
```

Lo que distingue a un agente de otro es: **su rol**, **qué campos lee**, **qué campo escribe** y **cuántos tokens puede usar**.

| Nodo | Lee | Escribe | `max_tokens` |
|---|---|---|---|
| `escritor` | tema, requisitos, borrador, metarevision | borrador, version | 350 |
| `revisor_seo` | borrador | seo | 140 |
| `revisor_legal` | borrador | legal | 140 |
| `revisor_etico` | borrador | etica | 140 |
| `metarrevisor` | seo, legal, etica | metarevision | 180 |
| `evaluador` | requisitos, borrador, metarevision | veredicto | 200 |

### 5.1 Escritor: redacta y reescribe
El mismo nodo cumple dos funciones según el estado:
- **Primera vez** (`borrador` vacío, `metarevision` vacía) → escribe desde cero.
- **Siguientes veces** → recibe el borrador anterior y el feedback consolidado, y lo reescribe.

Por eso el prompt envía `'NINGUNO'` cuando un campo está vacío: así el modelo sabe explícitamente que no hay material previo.

In [ ]:
ROL_ESCRITOR = "Eres un redactor de artículos de blog. Devuelve solo título y texto final; sé preciso."

def escritor(estado: EstadoBlog):
    # 1) Construir la tarea con todo lo que el escritor necesita saber.
    tarea = (f"Tema: {estado['tema']}\n"
             f"Requisitos: {estado['requisitos']}\n"
             f"Borrador previo: {estado['borrador'] or 'NINGUNO'}\n"
             f"Feedback consolidado: {estado['metarevision'] or 'NINGUNO'}")
    # 2) Llamar al modelo (es el agente con más tokens: produce el texto completo).
    texto = pedir_llm("escritor", ROL_ESCRITOR, tarea, max_tokens=350)
    # 3) Devolver SOLO los campos que este nodo posee.
    return {"borrador": texto,
            "version": estado["version"] + 1,          # 1ª, 2ª o 3ª versión
            "historial": registrar(estado, "escritor")}

### 5.2 Los tres revisores especializados
Los tres reciben **exactamente el mismo insumo** (el borrador) pero con **roles distintos**. Cada rol incluye restricciones explícitas:
- **SEO**: máximo tres recomendaciones y *no reescribir* (no debe invadir el trabajo del Escritor).
- **Legal**: señalar afirmaciones no verificadas y riesgos de privacidad, y *no afirmar que el texto es legalmente conforme*.
- **Ético**: sesgo, privacidad y responsabilidad.

🤔 **Pausa:** si los tres usan el mismo modelo y el mismo borrador, ¿qué ganamos al separarlos en tres nodos en lugar de un solo “revisor general”? ¿Qué perdemos?

In [ ]:
# Roles como constantes: así es fácil compararlos y ajustarlos sin tocar la lógica.
ROL_SEO = "Eres experto SEO. Devuelve máximo tres recomendaciones concretas; no reescribas el blog."
ROL_LEGAL = ("Eres revisor preliminar de riesgos legales de contenido. "
             "Señala afirmaciones no verificadas y riesgos de privacidad en máximo tres puntos; "
             "no afirmes que el texto es legalmente conforme.")
ROL_ETICO = ("Eres revisor de ética en IA. Detecta riesgos de sesgo, privacidad y "
             "responsabilidad en máximo tres puntos.")

def revisor_seo(estado: EstadoBlog):
    # Lee solo el borrador y escribe solo el campo "seo".
    comentario = pedir_llm("seo", ROL_SEO, estado["borrador"], max_tokens=140)
    return {"seo": comentario, "historial": registrar(estado, "seo")}

def revisor_legal(estado: EstadoBlog):
    # Mismo patrón, otra especialidad: escribe solo el campo "legal".
    comentario = pedir_llm("legal", ROL_LEGAL, estado["borrador"], max_tokens=140)
    return {"legal": comentario, "historial": registrar(estado, "legal")}

def revisor_etico(estado: EstadoBlog):
    # Mismo patrón, otra especialidad: escribe solo el campo "etica".
    comentario = pedir_llm("etica", ROL_ETICO, estado["borrador"], max_tokens=140)
    return {"etica": comentario, "historial": registrar(estado, "etica")}

### 5.3 Metarrevisor: consolidar sin redundancias
No lee el borrador, sino los **tres comentarios**. Su función es de coordinación: reducir ruido y priorizar, para que el Escritor reciba **un solo mensaje claro** en lugar de tres listas que pueden repetirse o contradecirse.

In [ ]:
ROL_META = "Eres metarrevisor. Consolida feedback sin redundancias en máximo 4 puntos prioritarios."

def metarrevisor(estado: EstadoBlog):
    # La tarea junta las tres revisiones, etiquetadas por especialidad.
    tarea = f"SEO: {estado['seo']}\nLegal: {estado['legal']}\nÉtica: {estado['etica']}"
    consolidado = pedir_llm("metarrevisor", ROL_META, tarea, max_tokens=180)
    return {"metarevision": consolidado, "historial": registrar(estado, "metarrevisor")}

### 5.4 Evaluador y función de ruta: dónde se toma la decisión
Aquí hay **dos piezas distintas** y conviene no confundirlas:

| Pieza | Tipo | Qué hace |
|---|---|---|
| `evaluador` | Nodo con LLM | Opina: escribe `APROBAR` o `REVISAR` + justificación en el estado |
| `ruta_evaluacion` | Función Python **determinista** (sin LLM) | Decide la siguiente arista leyendo el veredicto y el número de versión |

> 🧠 El LLM **recomienda**; el código **decide** y además impone el límite de versiones. Así, aunque el modelo pida `REVISAR` para siempre, el bucle termina.

> ⚠️ Un modelo puede responder `**REVISAR**` (con formato Markdown) o con un saludo antes de la palabra clave. Por eso extraemos el veredicto con una función tolerante (`extraer_veredicto`) en lugar de confiar en que la respuesta empiece exactamente con la palabra.

In [ ]:
ROL_EVALUADOR = ("Eres evaluador de calidad. Devuelve APROBAR o REVISAR al comienzo "
                 "y después una justificación breve. No inventes evidencia.")
MAX_VERSIONES = 3   # 1 redacción inicial + hasta 2 reescrituras

def evaluador(estado: EstadoBlog):
    tarea = (f"Requisitos: {estado['requisitos']}\n"
             f"Texto: {estado['borrador']}\n"
             f"Revisiones: {estado['metarevision']}")
    evaluacion = pedir_llm("evaluador", ROL_EVALUADOR, tarea, max_tokens=200)
    return {"veredicto": evaluacion, "historial": registrar(estado, "evaluador")}


def extraer_veredicto(texto: str) -> str:
    # Devuelve "APROBAR", "REVISAR" o "SIN_VEREDICTO".
    # Busca la PRIMERA palabra clave que aparezca, ignorando mayúsculas y formato.
    t = texto.upper()
    posiciones = {p: t.find(p) for p in ("APROBAR", "REVISAR") if p in t}
    if not posiciones:
        return "SIN_VEREDICTO"
    return min(posiciones, key=posiciones.get)


def ruta_evaluacion(estado: EstadoBlog) -> str:
    # Función de la arista condicional: devuelve una CLAVE ("reescribir" o "fin")
    # que el grafo traduce a un nodo destino (ver add_conditional_edges en la sección 6).
    decision = extraer_veredicto(estado["veredicto"])
    if decision == "REVISAR" and estado["version"] < MAX_VERSIONES:
        return "reescribir"
    # Se termina si: APROBAR, límite de versiones alcanzado o veredicto ilegible.
    # Ojo: terminar NO significa aprobar; el veredicto queda en el estado.
    return "fin"

### ▶️ 5.5 Probar la lógica de ruta sin gastar tokens
La función de ruta es **código puro**, así que podemos probarla con estados de mentira. Esta es una buena práctica: probar las decisiones de coordinación por separado de las respuestas del LLM.

In [ ]:
import pandas as pd

casos = [
    ("APROBAR: cumple requisitos",          1),
    ("REVISAR: falta privacidad",           1),
    ("REVISAR: falta privacidad",           3),   # límite alcanzado
    ("**REVISAR** el tono",                 2),   # formato Markdown
    ("Mi veredicto es APROBAR",             2),   # la palabra no va al inicio
    ("",                                    1),   # respuesta vacía
]
filas = []
for veredicto, version in casos:
    estado_falso = {"veredicto": veredicto, "version": version}   # solo los campos que lee la ruta
    filas.append({"veredicto": veredicto or "(vacío)", "version": version,
                  "extraído": extraer_veredicto(veredicto),
                  "ruta": ruta_evaluacion(estado_falso)})
display(pd.DataFrame(filas))

🤔 **Pausa:** en el último caso (respuesta vacía) el flujo termina con `SIN_VEREDICTO`. ¿Es la decisión correcta? ¿Qué harías en un sistema real: reintentar, escalar a una persona o rechazar la publicación?

### ▶️ 5.6 (Opcional) Probar un nodo aislado
Un nodo es solo una función: recibe un estado y devuelve **un diccionario parcial**. Ejecutarlo solo ayuda a entender qué devuelve antes de conectarlo al grafo.

> En modo real esta celda consume **1 llamada** al LLM.

In [ ]:
estado_prueba = {
    "tema": TEMA, "requisitos": REQUISITOS,
    "borrador": RESPUESTAS_SIMULADAS["escritor"],   # un borrador de ejemplo
    "seo": "", "legal": "", "etica": "", "metarevision": "",
    "version": 1, "veredicto": "", "historial": [],
}
salida = revisor_seo(estado_prueba)
print("Claves que devuelve el nodo:", list(salida.keys()))   # solo 'seo' e 'historial'
print("\nComentario SEO:\n", salida["seo"])
LLAMADAS.clear()   # no contamos esta prueba en las métricas de la sección 7

## 6. 🧠 Orquestación explícita con LangGraph

Construir el grafo tiene 4 pasos:

| Paso | Método | Qué significa |
|---|---|---|
| 1 | `StateGraph(EstadoBlog)` | Crear un grafo cuyo estado tiene la forma de `EstadoBlog` |
| 2 | `add_node(nombre, función)` | Registrar cada agente con un nombre |
| 3 | `add_edge(a, b)` | Arista **fija**: después de `a` siempre va `b` |
| 3b | `add_conditional_edges(a, f, mapa)` | Arista **condicional**: `f(estado)` devuelve una clave y `mapa` la traduce a un destino |
| 4 | `compile()` | Validar el grafo y obtener una aplicación ejecutable |

`START` y `END` son nodos especiales que marcan la entrada y la salida.

In [ ]:
# ── Paso 1: crear el constructor del grafo con el tipo de estado ─────────────
builder = StateGraph(EstadoBlog)

# ── Paso 2: registrar los nodos (nombre en el grafo → función de la sección 5) ─
builder.add_node("escritor", escritor)
builder.add_node("seo", revisor_seo)
builder.add_node("legal", revisor_legal)
builder.add_node("etica", revisor_etico)
builder.add_node("metarrevisor", metarrevisor)
builder.add_node("evaluador", evaluador)

# ── Paso 3: aristas fijas (el orden que SIEMPRE se cumple) ───────────────────
builder.add_edge(START, "escritor")        # todo empieza en el Escritor
builder.add_edge("escritor", "seo")        # revisión secuencial: SEO → Legal → Ética
builder.add_edge("seo", "legal")
builder.add_edge("legal", "etica")
builder.add_edge("etica", "metarrevisor")  # se consolidan las tres revisiones
builder.add_edge("metarrevisor", "evaluador")

# ── Paso 3b: arista condicional (el único punto de decisión del grafo) ───────
builder.add_conditional_edges(
    "evaluador",                 # desde qué nodo se decide
    ruta_evaluacion,             # función que devuelve una clave
    {"reescribir": "escritor",   # clave → nodo destino
     "fin": END},
)

# ── Paso 4: compilar ─────────────────────────────────────────────────────────
app = builder.compile()
print("Grafo compilado. Nodos:", [n for n in app.get_graph().nodes if not n.startswith("__")])

### ▶️ 6.1 Visualizar el grafo
LangGraph genera el diagrama a partir del grafo **compilado**, así que muestra lo que realmente se va a ejecutar (no lo que creemos haber programado).

**Cómo leerlo:** las líneas continuas son aristas fijas; las **punteadas** salen del evaluador y son condicionales (`reescribir` o `fin`).

> ⚠️ El PNG se genera con un servicio externo. Si falla (sin red o servicio caído), se imprime el código Mermaid, que puedes pegar en https://mermaid.live.

In [ ]:
try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("No fue posible renderizar PNG; aquí está el diagrama Mermaid:\n")
    print(app.get_graph().draw_mermaid())
    print("Detalle:", exc)

## 7. Ejecución completa y evidencia

### ▶️ 7.1 Ejecutar el grafo
- `estado_inicial` debe tener **todos** los campos del `EstadoBlog`; los que aún no tienen contenido van vacíos.
- `invoke()` ejecuta el grafo de principio a fin y devuelve el **estado final**.
- `recursion_limit=30` es un **guardrail** de LangGraph: si por un error de diseño el grafo diera más de 30 pasos, se detiene con error. Nuestro peor caso son 18 pasos.

In [ ]:
estado_inicial: EstadoBlog = {
    "tema": TEMA, "requisitos": REQUISITOS,
    "borrador": "", "seo": "", "legal": "", "etica": "", "metarevision": "",
    "version": 0,          # el escritor lo convertirá en 1
    "veredicto": "", "historial": [],
}

LLAMADAS.clear()              # empezar las métricas desde cero
_evaluaciones_simuladas = 0   # reiniciar el simulador (solo afecta al modo simulado)

inicio = time.perf_counter()
resultado = app.invoke(estado_inicial, config={"recursion_limit": 30})
latencia = time.perf_counter() - inicio

print("Versiones redactadas:", resultado["version"])
print("Ruta ejecutada:", " → ".join(resultado["historial"]))
print("Tiempo total:", round(latencia, 2), "segundos")
print("Veredicto final:", extraer_veredicto(resultado["veredicto"]))
print("Justificación:", resultado["veredicto"])
display(Markdown("### Artículo resultante\n" + resultado["borrador"]))

### ▶️ 7.2 ¿Qué dijo cada agente?
El estado final conserva el **último** comentario de cada especialista (los de iteraciones anteriores se sobrescribieron). Revísalos para comprobar si el artículo final realmente incorporó el feedback.

In [ ]:
for campo in ["seo", "legal", "etica", "metarevision"]:
    display(Markdown(f"**{campo.upper()}**\n\n{resultado[campo] or '_(vacío)_'}"))

palabras = len(resultado["borrador"].split())
print(f"\nComprobación determinista: el artículo tiene {palabras} palabras "
      f"({'cumple' if palabras <= 130 else 'NO cumple'} el límite de 130, contando también el título).")

> 🧠 La comprobación de palabras es **código**, no opinión de un agente. Siempre que un requisito se pueda verificar de forma determinista, conviene hacerlo así en lugar de confiar en el Evaluador.

### ▶️ 7.3 Costo de coordinación: llamadas y latencia por agente
Esta tabla es la **evidencia** para la discusión de costo. Compárala con el cálculo teórico: 6 llamadas por iteración.

In [ ]:
df_llamadas = pd.DataFrame(LLAMADAS)
display(df_llamadas)

resumen = (df_llamadas.groupby("agente", sort=False)
           .agg(llamadas=("agente", "size"), segundos=("segundos", "sum"))
           .round(2))
display(resumen)

iteraciones = resultado["version"]
print(f"Llamadas totales: {len(df_llamadas)} (esperadas: 6 × {iteraciones} iteración(es) = {6 * iteraciones})")
print(f"Tiempo en llamadas LLM: {df_llamadas['segundos'].sum():.2f} s de {latencia:.2f} s totales")

⚠️ **Una ejecución no es una medición.** La latencia varía con la red, la carga del servicio y los tokens generados. Para comparar contra un agente único se necesitan **al menos 3 repeticiones** por variante (lo harás en el notebook 01).

### ▶️ 7.4 (Opcional) Ver la ejecución nodo por nodo con `stream()`
`invoke()` solo entrega el estado final. `stream(..., stream_mode="updates")` entrega **lo que devuelve cada nodo en el momento en que termina**, ideal para explicar el flujo en vivo.

> En modo real esta celda vuelve a ejecutar todo el grafo (hasta 18 llamadas). Por eso solo corre automáticamente en modo simulado; cambia `VER_PASO_A_PASO = True` para forzarla.

In [ ]:
VER_PASO_A_PASO = True #MODO_SIMULADO

if VER_PASO_A_PASO:
    _evaluaciones_simuladas = 0
    paso = 0
    for actualizacion in app.stream(estado_inicial, config={"recursion_limit": 30},
                                    stream_mode="updates"):
        # Cada "actualizacion" es {nombre_del_nodo: {campos que ese nodo devolvió}}
        for nodo, cambios in actualizacion.items():
            paso += 1
            campos = [c for c in cambios if c != "historial"]
            # Recortamos el texto a 60 caracteres y quitamos saltos de línea para que quepa en una línea.
            vista = {c: str(cambios[c]).replace("\n", " ")[:60] + ("…" if len(str(cambios[c])) > 60 else "")
                     for c in campos}
            print(f"Paso {paso:>2} · {nodo:<13} actualizó {vista}")
else:
    print("Omitido para no consumir tokens. Cambia VER_PASO_A_PASO = True para ejecutarlo.")

## 8. Análisis arquitectónico: leer antes de copiar

### ¿Por qué es multiagéntico?
Hay **seis roles** que realizan inferencias independientes, con objetivos distintos, y colaboran a través de un **estado compartido**.

### ¿Qué lo orquesta?
El `StateGraph`: sus nodos, aristas fijas y **una** arista condicional. Es **orquestación estática** con un bucle de reflexión acotado.

### ¿Qué NO tiene aún?
| Ausente | Qué haría falta |
|---|---|
| Paralelismo entre revisores | Varias aristas desde `escritor` (fan-out) y reducers en los campos compartidos |
| Supervisor con delegación dinámica | Un nodo que decida con un LLM a quién llamar (`Command`) |
| Persistencia | Un *checkpointer* para guardar y reanudar ejecuciones |
| Aprobación humana | `interrupt()` + checkpointer antes de publicar |
| Herramientas externas | Búsqueda, verificación de fuentes, corrector, etc. |
| Validación jurídica real | Una persona experta: el agente legal solo señala riesgos de redacción |

### Costo y precisión
- Una iteración realiza **6 llamadas** LLM; cada nueva iteración añade otras 6 (peor caso: 18).
- **Más agentes no implica resultados más correctos.** La calidad debe evaluarse con evidencia externa y repeticiones, no con la opinión del propio Evaluador.

### 🤔 Diálogo en clase
1. ¿Qué revisores podrían ejecutarse en **paralelo**? ¿Cómo manejarías las escrituras concurrentes sobre `historial`?
2. ¿Es necesaria **siempre** la revisión legal? ¿Podría el grafo decidirlo según el tipo de contenido?
3. ¿Cómo **detendrías** el proceso si un revisor detecta un riesgo crítico?
4. ¿Qué **métricas** compararías contra un solo agente redactor?
5. El Evaluador juzga el borrador **antes** de que el Escritor incorpore el feedback. ¿Qué cambiaría si el orden fuera Metarrevisor → Escritor → Evaluador?

### ⚠️ Errores frecuentes al ejecutar
| Síntoma | Causa probable | Qué hacer |
|---|---|---|
| `ValueError: Configura GROQ_API_KEY` | El secreto no existe o no tiene acceso | Revisar el paso 2 (nombre exacto y acceso activado) |
| Error 413 o 429 de Groq | Se superó el límite de tokens o de peticiones por minuto | Esperar, reducir `max_tokens`, no relanzar en bucle |
| `⚠️ respuesta vacía` | El modelo agotó los tokens razonando | Subir `max_tokens` de ese agente |
| No aparece la imagen del grafo | Falla el servicio de renderizado | Usar el texto Mermaid impreso |
| Conflicto de versiones de pandas | Colab trae otra versión | Reiniciar el entorno después de la instalación |

## 9. Nuevo caso (actividad individual o por equipos): TRANSFERENCIA

**No se permite entregar solamente el mismo grafo con nombres de agentes diferentes.** Selecciona un problema diferente y explica qué coordinación aporta valor.

**Opciones de ejemplo:** un comité de evaluación de propuestas de proyectos; un equipo de planificación de evento académico; análisis de requerimientos de software; preparación de un plan de comunicación de crisis. Puedes proponer otro, pero debe ser distinto del blog.

### Entregables
1. **ADR de máximo 2 páginas**, con problema, mínimo dos alternativas, decisión, riesgos, justificación y diagrama.
2. **Notebook con al menos 3 agentes LLM de especialización distinta y una decisión de coordinación** (no solo ejecutar tareas secuenciales).
3. **Diagrama de secuencia** indicando qué información comparte cada agente.
4. **Tres casos de prueba reproducibles**, incluyendo información insuficiente y contradicción entre agentes.
5. **Análisis** de latencia, llamadas LLM y calidad respecto a agente único.

### Rúbrica
- Justificación arquitectónica: **30 %**
- Coordinación efectiva entre agentes: **30 %**
- Contratos y pruebas: **25 %**
- Análisis de resultados: **15 %**

**Fecha orientativa de entrega:** 20 de octubre, 23:59, si es la fecha confirmada en Classroom.

### ✅ Lista de verificación antes de entregar
- [ ] Mi problema es distinto del blog y explico por qué necesita varios agentes.
- [ ] Tengo una tabla de **quién lee y quién escribe** cada campo del estado.
- [ ] Hay al menos **una decisión de coordinación** (arista condicional, router o supervisor).
- [ ] El bucle, si existe, tiene un **límite** explícito.
- [ ] Separé lo que decide el **LLM** de lo que decide el **código**.
- [ ] Medí llamadas y latencia con **al menos 3 repeticiones**.
- [ ] No incluí la clave de Groq en el notebook.

## 10. Fuentes y aclaraciones
- Notebook de referencia suministrado por la docente: `L3_Reflection_and_Blogpost_Writing.ipynb` (ejemplo original en AutoGen).
- LangGraph: https://docs.langchain.com/oss/python/langgraph/overview
- Grafo de estado y API: https://reference.langchain.com/python/langgraph/graph/state/StateGraph
- Groq: https://console.groq.com/docs/model/openai/gpt-oss-120b

*Esta adaptación utiliza una implementación propia. No reproduce las llamadas de AutoGen del notebook original.*